In [2]:
feature_names = joblib.load("outputs/saved_models/kaggle_stylometric_feature_names.pkl")

print(feature_names)

['n_characters', 'n_characters_no_spaces', 'n_words', 'n_sentences', 'mean_word_length', 'std_word_length', 'mean_sentence_length', 'std_sentence_length', 'lexical_diversity', 'repetition_rate', 'uppercase_share', 'digit_share', 'stopword_share', 'n_em_dash', 'has_em_dash', 'n_ellipsis', 'flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog', 'count_.', 'count_,', 'count_;', 'count_:', 'count_!', 'count_?', 'count_"', "count_'", 'count_(', 'count_)', 'count_-', 'rate_.', 'rate_,', 'rate_;', 'rate_:', 'rate_!', 'rate_?', 'rate_"', "rate_'", 'rate_(', 'rate_)', 'rate_-', 'pos_share_noun', 'pos_share_verb', 'pos_share_adj', 'pos_share_adv', 'pos_share_pron', 'pos_share_det', 'pos_share_adp', 'pos_share_aux', 'pos_share_cconj', 'pos_share_sconj', 'pos_share_propn', 'pos_share_num']


In [1]:
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import f1_score, recall_score, accuracy_score

from stylometric_features import compute_stylometric_features

# =========================================================
# 1. LOAD EXTERNAL DATA
# =========================================================
df = pd.read_csv("raw_data/Medium_data.csv") 

# human sources
human_sources = {"guardian", "arxiv"}

# binary label: 1 = AI, 0 = human
df["generated"] = (~df["source"].isin(human_sources)).astype(int)

X_text = df["text"].astype(str)
y_true = df["generated"].astype(int).values


# =========================================================
# 2. HELPERS
# =========================================================
def predict_with_threshold(proba, threshold):
    return (proba >= threshold).astype(int)

def evaluate_overall(y_true, y_pred):
    return {
        "overall_f1": f1_score(y_true, y_pred, zero_division=0),
        "ai_recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }

def share_predicted_ai_by_source(df, pred_col, human_sources={"guardian", "arxiv"}):
    out = []
    ai_df = df[~df["source"].isin(human_sources)].copy()

    for src, g in ai_df.groupby("source"):
        out.append({
            "source": src,
            "n": len(g),
            "share_predicted_ai": g[pred_col].mean()
        })

    return pd.DataFrame(out).sort_values("source").reset_index(drop=True)


# =========================================================
# 3. TF-IDF MODEL
# =========================================================
tfidf_pipeline = joblib.load("outputs/saved_models/kaggle_tfidf_logreg_pipeline.pkl")
threshold_tfidf = joblib.load("outputs/saved_models/kaggle_threshold_tfidf.pkl")

proba_tfidf = tfidf_pipeline.predict_proba(X_text)[:, 1]
df["pred_tfidf"] = predict_with_threshold(proba_tfidf, threshold_tfidf)

res_tfidf = evaluate_overall(y_true, df["pred_tfidf"].values)


# =========================================================
# 4. STYLOMETRIC MODEL
# =========================================================
stylometric_model = joblib.load("outputs/saved_models/kaggle_stylometric_logreg.pkl")
threshold_stylo = joblib.load("outputs/saved_models/kaggle_threshold_stylometric.pkl")
stylo_feature_names = joblib.load("outputs/saved_models/kaggle_stylometric_feature_names.pkl")

X_stylo = pd.DataFrame(X_text.apply(compute_stylometric_features).tolist())
X_stylo = X_stylo[stylo_feature_names]

proba_stylo = stylometric_model.predict_proba(X_stylo)[:, 1]
df["pred_stylo"] = predict_with_threshold(proba_stylo, threshold_stylo)

res_stylo = evaluate_overall(y_true, df["pred_stylo"].values)


# =========================================================
# 5. COMBINED MODEL
# =========================================================
combined_model = joblib.load("outputs/saved_models/kaggle_combined_logreg.pkl")
combined_vectorizer = joblib.load("outputs/saved_models/kaggle_combined_tfidf_vectorizer.pkl")
threshold_combined = joblib.load("outputs/saved_models/kaggle_threshold_combined.pkl")
combined_stylo_feature_names = joblib.load("outputs/saved_models/kaggle_combined_stylometric_feature_names.pkl")

X_tfidf_combined = combined_vectorizer.transform(X_text)

X_stylo_combined = pd.DataFrame(X_text.apply(compute_stylometric_features).tolist())
X_stylo_combined = X_stylo_combined[combined_stylo_feature_names]

X_combined = hstack([
    X_tfidf_combined,
    csr_matrix(X_stylo_combined.values)
])

proba_combined = combined_model.predict_proba(X_combined)[:, 1]
df["pred_combined"] = predict_with_threshold(proba_combined, threshold_combined)

res_combined = evaluate_overall(y_true, df["pred_combined"].values)


# =========================================================
# 6. SUMMARY TABLE
# =========================================================
summary = pd.DataFrame([
    {"model": "tfidf_logreg", **res_tfidf},
    {"model": "stylometric_logreg", **res_stylo},
    {"model": "combined_logreg", **res_combined},
]).sort_values("overall_f1", ascending=False)

print("\n=== OVERALL RESULTS ===")
print(summary.round(4))


# =========================================================
# 7. SHARE PREDICTED AS AI BY AI SOURCE
# =========================================================
share_source_tfidf = share_predicted_ai_by_source(df, "pred_tfidf").rename(
    columns={"share_predicted_ai": "share_pred_ai_tfidf"}
)
share_source_stylo = share_predicted_ai_by_source(df, "pred_stylo").rename(
    columns={"share_predicted_ai": "share_pred_ai_stylo"}
)
share_source_combined = share_predicted_ai_by_source(df, "pred_combined").rename(
    columns={"share_predicted_ai": "share_pred_ai_combined"}
)

share_by_source_all = (
    share_source_tfidf
    .merge(share_source_stylo[["source", "share_pred_ai_stylo"]], on="source")
    .merge(share_source_combined[["source", "share_pred_ai_combined"]], on="source")
)

print("\n=== SHARE PREDICTED AS AI BY AI SOURCE ===")
print(share_by_source_all.round(4))

KeyError: '[\'count_.\', \'count_,\', \'count_;\', \'count_:\', \'count_!\', \'count_?\', \'count_"\', "count_\'", \'count_(\', \'count_)\', \'count_-\', \'rate_.\', \'rate_,\', \'rate_;\', \'rate_:\', \'rate_!\', \'rate_?\', \'rate_"\', "rate_\'", \'rate_(\', \'rate_)\', \'rate_-\'] not in index'